In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Загрузка модели и изображения
net = cv2.dnn.readNetFromONNX(r"G:\Work\Fish-Guard_2_0\ML\new_colab.onnx")
image = cv2.imread(r"F:\Datasets\Reddit-fishes\aquarium_top_2\gallery-dl\reddit\Aquariums\1hynu9g 04 Welcome fishie !! Help give me names _) (funny ones) .jpg")
original = image.copy()
h, w = image.shape[:2]

# Препроцессинг
blob = cv2.dnn.blobFromImage(image, 1/255.0, (960, 960), swapRB=True)
net.setInput(blob)
outputs = net.forward()

# ДИАГНОСТИКА - посмотрим что возвращает модель
print("Shape of outputs:", outputs.shape)  # Должно быть (1, 5, 18900)
print("Sample values:", outputs[0, :, 0])  # Первый детекшн

# Постобработка для YOLO с одним классом
def process_output(outputs, conf_threshold=0.5, nms_threshold=0.4):
    # outputs shape: (1, 5, 18900) где 5: [cx, cy, w, h, confidence]
    predictions = np.squeeze(outputs).T  # Транспонируем к (18900, 5)
    
    # Отфильтровываем по confidence
    scores = predictions[:, 4]  # Confidence scores
    mask = scores > conf_threshold
    predictions = predictions[mask]
    scores = scores[mask]
    
    if len(scores) == 0:
        return []
    
    # Конвертируем из нормализованных координат в пиксельные
# Конвертируем из нормализованных координат (относительно 960x960) в пиксельные исходного изображения
    boxes = predictions[:, :4]
    scale_x = w / 960  # Масштаб по X
    scale_y = h / 960  # Масштаб по Y

    x1 = (boxes[:, 0] - boxes[:, 2] / 2) * scale_x
    y1 = (boxes[:, 1] - boxes[:, 3] / 2) * scale_y  
    x2 = (boxes[:, 0] + boxes[:, 2] / 2) * scale_x
    y2 = (boxes[:, 1] + boxes[:, 3] / 2) * scale_y
    boxes = np.column_stack([x1, y1, x2, y2])
    
    # Применяем NMS
    indices = cv2.dnn.NMSBoxes(boxes.tolist(), scores.tolist(), conf_threshold, nms_threshold)
    
    if len(indices) > 0:
        return [(boxes[i], scores[i]) for i in indices.flatten()]
    return []

# Обрабатываем выходы
detections = process_output(outputs)

# Рисуем bounding boxes
for box, score in detections:
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 3)
    label = f"Fish: {score:.2f}"
    cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0, 255, 0), 5)

# Отображаем результат
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

print(f"Найдено рыб: {len(detections)}")
for i, (box, score) in enumerate(detections):
    print(f"Рыба {i+1}: confidence={score:.2f}, bbox={box.astype(int)}")

In [ ]:
import onnx
import numpy as np

# Загрузка и анализ ONNX модели
model = onnx.load(r"G:\Work\Fish_Demo\fish-guard-ml\detect.onnx")

print("Входы модели:")
for inp in model.graph.input:
    print(f"  - {inp.name}: {inp.type}")
    
print("\nВыходы модели:")
for out in model.graph.output:
    print(f"  - {out.name}: {out.type}")

# Посмотрим на shape выходов
for out in model.graph.output:
    shape = []
    for dim in out.type.tensor_type.shape.dim:
        if dim.dim_value:
            shape.append(dim.dim_value)
        elif dim.dim_param:
            shape.append(dim.dim_param)
        else:
            shape.append('?')
    print(f"  {out.name} shape: {shape}")

In [ ]:
import numpy as np

# Посмотрим форму и содержимое вывода
print("Shape of outputs:", outputs.shape)
print("Number of dimensions:", outputs.ndim)

# Если вывод многомерный, посмотрим первые элементы
if outputs.ndim > 1:
    print("First few elements:", outputs[0, :5, 0])  # Первые 5 элементов первого detection

In [ ]:
# ДИАГНОСТИКА - детальный анализ outputs
print("Shape of outputs:", outputs.shape)
print("Min/Max values in outputs:", np.min(outputs), np.max(outputs))

# Посмотрим на confidence scores
predictions = np.squeeze(outputs).T
scores = predictions[:, 4]
print("Confidence scores stats:")
print("  Min:", np.min(scores))
print("  Max:", np.max(scores)) 
print("  Mean:", np.mean(scores))
print("  Number of scores > 0.5:", np.sum(scores > 0.5))
print("  Number of scores > 0.1:", np.sum(scores > 0.1))
print("  Number of scores > 0.01:", np.sum(scores > 0.01))

# Посмотрим первые 10 детекций
print("\nFirst 10 detections (cx, cy, w, h, conf):")
for i in range(min(10, len(predictions))):
    print(f"  {i}: {predictions[i]}")